In [51]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from pymilvus import MilvusClient

from huggingface_hub import InferenceClient

from dotenv import load_dotenv
from tqdm import tqdm
import json
import os

In [53]:
load_dotenv()
hf_token = os.environ["HF_TOKEN"]

In [10]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small", cache_folder=f'models_cache')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5153.85it/s]


In [54]:
def emb_text(text):
    return embedding_model.encode([text], normalize_embeddings=True).tolist()[0]

In [46]:
milvus_client = MilvusClient(uri="http://localhost:19530")

collection_name = "RAG_col"
metric_type = "IP"

In [47]:
question = "Какие задачи у помощников"

In [48]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=[
        emb_text(question)
    ],
    limit=5,
    search_params={"metric_type": "IP", "params": {}},
    output_fields=["text"],
)

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]

print(
    json.dumps(
        retrieved_lines_with_distances,
        indent=4,
        ensure_ascii=False
    )
)

[
    [
        "Что делают помощники ведущего, волонтёры? \n \nЗадача волонтёров мероприятия — сделать так, чтобы мероприятие прошло гладко и \nкомфортно и для участников, и для команды организаторов: расставить стулья, \nподготовить сертификаты победителей, зарегистрировать команды участников на \nмероприятие (например, какие-то команды могут не прийти на мероприятия и наоборот, \nкто-то захочет принять участие, не будучи зарегистрированным). Всё перечисленные \nвыше и многие другие организационные моменты находятся в ответственности волонтёра \nи должны быть продуманы и подготовлены заранее. \n \nЗадача помощников ведущего игр — быть дополнительными «руками и глазами» \nведущего, помогать с подготовкой раздаточного мероприятия, расстановкой реквизита на \nстанции и т. д. \n \nОбязательно ли собрать 80 участников для проведения мероприятия, как это указано в \nруководстве к проведению?",
        0.864365816116333
    ],
    [
        "• подробно изучить сценарий и все материалы для п

In [55]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

PROMPT = """
Используй следующие части информации закрытые в теге <context> для ответа на вопрос в теге <question>.
<context>
{context}
</context>
<question>
{question}
</question>
"""

In [56]:
llm_client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
    timeout=120
)

In [57]:
prompt = PROMPT.format(context=context, question=question)

completion = llm_client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct:featherless-ai",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
)

print(completion.choices[0].message.content)

Спасибо за предоставленную информацию. По вашему запросу:

**Какие задачи у помощников**

Разделенные на четыре основных этапа подготовки и проведения мероприятия, помощники ведущих игра выполняют следующие ключевые задачи:

### Этап 1: Подготовка к мероприятию

- **Получение списка задач от ведущих игр:** Помощники ведущих игр получают списки задач от ведущих игр для помощи во время проведения игр.
- **Получение списка задач от организатора по подготовке площадки, регистрации участников:** Они также получают списки задач от организатора, связанных с подготовкой площадки, регистрации участников и другими важными деталями.

### Этап 2: Подготовка к мероприятию

- **Принятие участия в генеральной репетиции:** Помощники принимают участие в генеральной репетиции, где они помогают в подготовке к проведению мероприятия.
- **Получение списка задач от ведущих игр для помощи во время проведения игр:** Они получают списки задач от ведущих игр для того, чтобы помочь им в процессе проведения игр.
